# Day 8: Hierarchy, Parameters & Generate

## Pre-Class Videos (~45 minutes total)

| # | Segment | Duration | File |
|---|---------|----------|------|
| 1 | Module Hierarchy Deep Dive | ~12 min | `d08_s1_module_hierarchy.html` |
| 2 | Parameters & Parameterization | ~15 min | `d08_s2_parameters_parameterization.html` |
| 3 | Generate Blocks | ~12 min | `d08_s3_generate_blocks.html` |
| 4 | Design for Reuse | ~6 min | `d08_s4_design_for_reuse.html` |

## Code Examples

| File | Description |
|------|-------------|
| `code/day08_ex01_parallel_debounce.v` | Generate-based N-button input pipeline (debounce + edge detect) |
| `code/day08_ex02_param_alu.v` | Parameterized N-bit ALU with self-checking testbench at WIDTH=4 and 8 |

## Diagrams

| File | Description |
|------|-------------|
| `diagrams/d08_hierarchy_tree.svg` | Module hierarchy tree with levels, parameterization annotations |
| `diagrams/d08_generate_concept.svg` | generate-for: one source → N hardware copies at elaboration |

## Key Concepts
- Hierarchy for complexity management, descriptive naming
- `parameter` (configurable) vs. `localparam` (internal/derived)
- `$clog2()` for automatic width sizing
- `generate for` — hardware replication at elaboration time
- `generate if` — conditional hardware inclusion
- Module reuse checklist: parameterized, tested, documented

## Week 2 Recap

Your module library after Week 2 (~10 tested, reusable modules):
`hex_to_7seg`, `debounce`, `counter_mod_n`, `edge_detector`, `synchronizer`,
`shift_reg_piso`, `fsm_template`, `pattern_detector`, `parallel_debounce`, `param_alu`

These are the building blocks for Week 3 (UART, SPI, memory).

## Directory Structure

```
day08_hierarchy_parameters_generate/
├── d08_s1_module_hierarchy.html
├── d08_s2_parameters_parameterization.html
├── d08_s3_generate_blocks.html
├── d08_s4_design_for_reuse.html
├── code/
│   ├── day08_ex01_parallel_debounce.v
│   └── day08_ex02_param_alu.v
├── diagrams/
│   ├── d08_hierarchy_tree.svg
│   └── d08_generate_concept.svg
├── day08_quiz.md
└── day08_readme.md
```

---
## Code Examples

### `day08_ex01_parallel_debounce.v`

```verilog
// =============================================================================
// day08_ex01_parallel_debounce.v — Generate-Based N-Button Input Pipeline
// Day 8: Hierarchy, Parameters & Generate
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================
// Demonstrates: generate-for, parameterized instantiation, hierarchy.
// Creates N independent debounce + edge-detect pipelines from one module.
// =============================================================================
// Synth:  yosys -p "read_verilog day08_ex01_parallel_debounce.v debounce.v; \
//          synth_ice40 -top parallel_debounce"
// =============================================================================

module parallel_debounce #(
    parameter N              = 4,          // Number of buttons
    parameter CLKS_TO_STABLE = 250_000     // Debounce threshold (10 ms at 25 MHz)
)(
    input  wire         i_clk,
    input  wire [N-1:0] i_buttons,      // raw button inputs (active low ok)
    output wire [N-1:0] o_clean,        // debounced levels
    output wire [N-1:0] o_press_edge,   // one-cycle pulse on press
    output wire [N-1:0] o_release_edge  // one-cycle pulse on release
);

    // ---- Generate N debounce + edge detect pipelines ----
    genvar g;
    generate
        for (g = 0; g < N; g = g + 1) begin : gen_input
            // Debouncer (includes built-in 2-FF synchronizer)
            debounce #(
                .CLKS_TO_STABLE(CLKS_TO_STABLE)
            ) db (
                .i_clk    (i_clk),
                .i_bouncy (i_buttons[g]),
                .o_clean  (o_clean[g])
            );

            // Edge detector
            reg r_prev;
            always @(posedge i_clk)
                r_prev <= o_clean[g];

            assign o_press_edge[g]   = o_clean[g] & ~r_prev;   // rising edge
            assign o_release_edge[g] = ~o_clean[g] & r_prev;   // falling edge
        end
    endgenerate

endmodule
```

### `day08_ex02_param_alu.v`

```verilog
// =============================================================================
// day08_ex02_param_alu.v — Parameterized N-Bit ALU
// Day 8: Hierarchy, Parameters & Generate
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================
// Refactored from the Day 3 hardcoded 4-bit ALU into a reusable,
// parameterized module. Demonstrates parameter, localparam, and $clog2.
// =============================================================================
// Build:  iverilog -DSIMULATION -o sim day08_ex02_param_alu.v && vvp sim
// Synth:  yosys -p "read_verilog day08_ex02_param_alu.v; synth_ice40 -top param_alu"
// =============================================================================

module param_alu #(
    parameter WIDTH = 8
)(
    input  wire [WIDTH-1:0] i_a,
    input  wire [WIDTH-1:0] i_b,
    input  wire [2:0]       i_opcode,
    output reg  [WIDTH-1:0] o_result,
    output reg              o_carry,
    output wire             o_zero
);

    // ---- Opcode Definitions ----
    localparam OP_ADD = 3'b000;
    localparam OP_SUB = 3'b001;
    localparam OP_AND = 3'b010;
    localparam OP_OR  = 3'b011;
    localparam OP_XOR = 3'b100;
    localparam OP_NOT = 3'b101;
    localparam OP_SHL = 3'b110;
    localparam OP_SHR = 3'b111;

    // ---- Combinational ALU ----
    reg [WIDTH:0] r_wide;   // extra bit for carry

    always @(*) begin
        r_wide = {(WIDTH+1){1'b0}};  // default

        case (i_opcode)
            OP_ADD: r_wide = {1'b0, i_a} + {1'b0, i_b};
            OP_SUB: r_wide = {1'b0, i_a} - {1'b0, i_b};
            OP_AND: r_wide = {1'b0, i_a & i_b};
            OP_OR:  r_wide = {1'b0, i_a | i_b};
            OP_XOR: r_wide = {1'b0, i_a ^ i_b};
            OP_NOT: r_wide = {1'b0, ~i_a};
            OP_SHL: r_wide = {i_a, 1'b0};          // shift left (carry = MSB)
            OP_SHR: r_wide = {i_a[0], 1'b0, i_a[WIDTH-1:1]};  // shift right
            default: r_wide = {(WIDTH+1){1'b0}};
        endcase

        o_result = r_wide[WIDTH-1:0];
        o_carry  = r_wide[WIDTH];
    end

    assign o_zero = (o_result == {WIDTH{1'b0}});

endmodule

// =============================================================================
// Self-Checking Testbench — tests at WIDTH=4 and WIDTH=8
// =============================================================================
`ifdef SIMULATION
module tb_param_alu;
    // ---- 4-bit instance ----
    reg  [3:0] a4, b4;
    reg  [2:0] op;
    wire [3:0] result4;
    wire       carry4, zero4;

    param_alu #(.WIDTH(4)) alu4 (
        .i_a(a4), .i_b(b4), .i_opcode(op),
        .o_result(result4), .o_carry(carry4), .o_zero(zero4)
    );

    // ---- 8-bit instance ----
    reg  [7:0] a8, b8;
    wire [7:0] result8;
    wire       carry8, zero8;

    param_alu #(.WIDTH(8)) alu8 (
        .i_a(a8), .i_b(b8), .i_opcode(op),
        .o_result(result8), .o_carry(carry8), .o_zero(zero8)
    );

    integer test_count = 0, fail_count = 0;

    task check4;
        input [3:0] exp;
        input [8*30-1:0] name;
    begin
        test_count = test_count + 1;
        if (result4 !== exp) begin
            $display("FAIL [4-bit]: %0s — expected %h, got %h", name, exp, result4);
            fail_count = fail_count + 1;
        end else
            $display("PASS [4-bit]: %0s = %h", name, result4);
    end
    endtask

    task check8;
        input [7:0] exp;
        input [8*30-1:0] name;
    begin
        test_count = test_count + 1;
        if (result8 !== exp) begin
            $display("FAIL [8-bit]: %0s — expected %h, got %h", name, exp, result8);
            fail_count = fail_count + 1;
        end else
            $display("PASS [8-bit]: %0s = %h", name, result8);
    end
    endtask

    initial begin
        $dumpfile("tb_param_alu.vcd");
        $dumpvars(0, tb_param_alu);
        $display("\n=== Parameterized ALU Testbench ===\n");

        // 4-bit tests
        op = 3'b000;  // ADD
        a4 = 4'd3; b4 = 4'd5; a8 = 8'd3; b8 = 8'd5;
        #100;
        check4(4'd8, "ADD 3+5");
        check8(8'd8, "ADD 3+5");

        a4 = 4'hF; b4 = 4'h1; a8 = 8'hFF; b8 = 8'h01;
        #100;
        check4(4'h0, "ADD F+1 overflow");
        check8(8'h0, "ADD FF+1 overflow");

        op = 3'b010;  // AND
        a4 = 4'hA; b4 = 4'hC; a8 = 8'hAA; b8 = 8'hCC;
        #100;
        check4(4'h8, "AND A&C");
        check8(8'h88, "AND AA&CC");

        op = 3'b101;  // NOT
        a4 = 4'h0; a8 = 8'h0;
        #100;
        check4(4'hF, "NOT 0=F");
        check8(8'hFF, "NOT 00=FF");

        // Summary
        $display("\n=== TEST SUMMARY ===");
        $display("Tests: %0d  Passed: %0d  Failed: %0d",
                 test_count, test_count - fail_count, fail_count);
        if (fail_count == 0)
            $display("\n*** ALL TESTS PASSED ***\n");
        else
            $display("\n*** %0d FAILURES ***\n", fail_count);
        $finish;
    end
endmodule
`endif
```

---
## Pre-Class Self-Check Quiz

**Q1:** What is the difference between `parameter` and `localparam`?

<details><summary>Answer</summary>

`parameter` can be overridden at instantiation time using the `#(.PARAM(value))` syntax — use for configurable values like widths, thresholds, and counts.
`localparam` is internal to the module and **cannot** be overridden — use for derived constants (e.g., `localparam MAX = (1 << WIDTH) - 1`).

</details>

**Q2:** Is `generate for` a runtime loop? Explain what it actually does.

<details><summary>Answer</summary>

No! `generate for` runs at **elaboration time** (compile time). It physically creates N independent copies of the hardware. Each iteration produces a separate instance with its own flip-flops and logic. It is NOT a sequential loop — it is hardware replication.

</details>

**Q3:** What does `$clog2(1000)` return and why is it useful?

<details><summary>Answer</summary>

It returns **10** — because ceil(log₂(1000)) = 10. You need 10 bits to represent values 0 through 999. It's useful because it automatically derives the correct bit width from a parameter, eliminating manual calculation and the bugs that come with it. When the parameter changes, the width adjusts automatically.

</details>

**Q4:** Why should you name generate blocks (e.g., `begin : gen_debounce`)?

<details><summary>Answer</summary>

Named generate blocks create hierarchical paths visible in simulation waveforms and synthesis reports. Debugging `gen_debounce[2].db.r_count` is much easier than navigating anonymous generated instances. It also makes `$display` and `$dumpvars` output meaningful.

</details>